# Phi-3.5V-Med 실행 (transformers 4.44.2): 베이스라인 확인 · RAG · Expert-CFG

모든 로직은 `phi35cfg/` 패키지에 있고 이 노트북은 환경 준비와 CLI 호출만 한다.
규칙 준수 행만 만든다(정답을 입력에 넣지 않는다). 채점은 원본(Phi) 방식이다.

1. 코드는 공개 레포의 `REPO_REF` 브랜치를 clone 한다(토큰 불필요). 실행한 커밋이 1절에 출력된다.
2. GPU 런타임에서 위에서부터 실행한다.
3. `DATASET` 을 바꾸면 런타임을 다시 시작하고 처음부터 실행한다.

## 0. 설정

In [ ]:
DATASET = 'vqarad'                     # 'vqarad' | 'slake'
SHARED = '/content/drive/MyDrive/AIMS_shared'
REPO_URL = 'https://github.com/MinseongKim-dev/AIMS_basement.git'
REPO_REF = 'phi35cfg-modularization'   # 브랜치 이름. 결과를 고정하려면 1절에 출력되는 커밋을 기록해 둔다
ADAPTER_NAME = {'vqarad': 'phi35-vqarad-e10-s42-v1', 'slake': 'phi35-slake-e5-s42-v1'}[DATASET]
BASE_NAME = {'vqarad': 'phi35-vqarad-baseline-ckpt257-v1', 'slake': 'phi35-slake-baseline-ckpt380-v1'}[DATASET]
IMAGE_ROOT = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'     # VQA-RAD 이미지 (SLAKE 는 쓰지 않음)
DATA_ROOT = '/content/slake_data'                                       # SLAKE 는 여기에 자동으로 받는다
RAG = f'/content/drive/MyDrive/ExpertCFG_replication/rag_{DATASET}_peir_roco_trainqa_v2.json'   # 검색 파일 (이전에 만든 위치)
TEAM_ENTROPY = f'{SHARED}/uncertainty/phi35-{DATASET}-eq4-v1/entropy.json'      # 팀원 파일 (있으면 이것이 기준)
OUT = f'/content/drive/MyDrive/phi35cfg_runs/{DATASET}'                # 결과 (끊겨도 이어서 실행)

BASE_DIR = '/content/Phi-3.5-vision-instruct'
ADAPTER = f'/content/releases/{ADAPTER_NAME}'
BASELINE = f'/content/releases/{BASE_NAME}/test_predictions.json'
CODE = '/content/AIMS_basement'

## 1. 코드 받기 · 환경

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, zipfile
os.chdir('/content')
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_REF, REPO_URL, CODE], check=True)   # git clone
print('code commit:', subprocess.check_output(['git', '-C', CODE, 'rev-parse', 'HEAD'], text=True).strip())   # git rev-parse HEAD
os.makedirs(OUT, exist_ok=True)
%cd {CODE}/phi35

In [ ]:
import sys, importlib.metadata as im
_before = sys.modules['numpy'].__version__ if 'numpy' in sys.modules else None
!pip install -q -r requirements.txt
!pip uninstall -y -q bitsandbytes
if _before is not None and _before != im.version('numpy'):
    print('numpy 가 바뀌어 런타임을 재시작합니다. 설정 셀부터 다시 실행하세요.')
    os.kill(os.getpid(), 9)
assert im.version('transformers') == '4.44.2', im.version('transformers')

In [ ]:
!python ../tools/fetch_author_assets.py
# 릴리스(어댑터 · 베이스라인 예측)를 복사한 뒤 해시 검증
for name, sub in ((ADAPTER_NAME, 'checkpoints/phi35'), (BASE_NAME, 'baselines')):
    shutil.copytree(f'{SHARED}/{sub}/{name}', f'/content/releases/{name}', dirs_exist_ok=True)
!python ../tools/release_manifest.py verify --name {ADAPTER_NAME} --path /content/releases/{ADAPTER_NAME}
!python ../tools/release_manifest.py verify --name {BASE_NAME} --path /content/releases/{BASE_NAME}
if DATASET == 'slake' and not os.path.isdir(f'{DATA_ROOT}/imgs'):
    from huggingface_hub import hf_hub_download
    zipfile.ZipFile(hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')).extractall(DATA_ROOT)

## 2. 검색 파일
이미 만든 파일을 `RAG` 에 지정하거나, `BUILD_RAG = True` 로 생성한다(PEIR 캡션 · 키워드는 저자 배포본이 필요하다).

In [ ]:
BUILD_RAG = False
PEIR_DIR = '/content/drive/MyDrive/ExpertCFG_replication'
if BUILD_RAG:
    if not os.path.isdir('/content/slake_data/imgs'):
        from huggingface_hub import hf_hub_download
        zipfile.ZipFile(hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')).extractall('/content/slake_data')
    !python ../common/build_rag_testjson.py --peir_dir "{PEIR_DIR}" --vqarad_common ../data_common/vqarad_img/testset.json --vqarad_train ../data_common/vqarad_img/trainset.json --vqarad_img "{IMAGE_ROOT}" --out_dir "{OUT}" --out_suffix _v2
    RAG = f'{OUT}/rag_{DATASET}_peir_roco_trainqa_v2.json'
assert os.path.exists(RAG), RAG

## 3. GPU 점검 (FAILED 가 나오면 아래를 실행하지 않는다)

In [ ]:
!python tests/gpu_check.py --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --retrieval "{RAG}" --predictions {BASELINE} --out "{OUT}/gpu_check.json"

## 4. 학습 · 체크포인트 선택 (선택 사항)
릴리스 어댑터를 쓰면 건너뛴다. `SMOKE_TRAIN` 은 몇 스텝 실행과 이어하기만 확인한다.

In [ ]:
SMOKE_TRAIN = False
RUN_TRAIN = False
TRAIN_DIR = f'{OUT}/train'
RESUME = ''                            # 끊긴 학습을 이어갈 때 '--resume'
if SMOKE_TRAIN:
    !python -m phi35cfg.train --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --output /content/train_smoke --attn sdpa --max-steps 2
    !python -m phi35cfg.train --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --output /content/train_smoke --attn sdpa --max-steps 4 --resume
if RUN_TRAIN:
    !python -m phi35cfg.train --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --output "{TRAIN_DIR}" --attn sdpa {RESUME}

In [ ]:
if RUN_TRAIN:                          # val 로만 고른다
    import glob
    steps = sorted(int(d.rsplit('-', 1)[1]) for d in glob.glob(f'{TRAIN_DIR}/checkpoint-*'))
    for step in steps:
        !python -m phi35cfg.infer --mode baseline --split val --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter "{TRAIN_DIR}/checkpoint-{step}" --output "{OUT}/val_{step}.jsonl"
    RUNS = ' '.join(f'"{s}={OUT}/val_{s}.jsonl"' for s in steps)
    !python -m phi35cfg.evaluate select --dataset {DATASET} --data-root "{DATA_ROOT}" --run {RUNS} --out "{OUT}/selected.json"
    # 새 어댑터를 평가하려면 선택된 체크포인트로 test 베이스라인을 만들고 ADAPTER · BASELINE · ADAPTER_NAME · BASE_NAME 을 바꾼다

## 5. 불확실성 (식 4) — 팀원 파일이 있으면 그것을 쓴다

In [ ]:
if os.path.exists(TEAM_ENTROPY):
    ENTROPY = TEAM_ENTROPY
else:
    ENTROPY = f'{OUT}/entropy.json'
    !python -m phi35cfg.infer --mode entropy --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --adapter-name {ADAPTER_NAME} --predictions {BASELINE} --predictions-name {BASE_NAME} --output "{ENTROPY}"
print(ENTROPY)

## 6. 실험 (규칙 준수 행)

In [ ]:
# 4.44.2 에서 새로 만든 베이스라인 (확인용 행. 기준은 릴리스 예측)
!python -m phi35cfg.infer --mode baseline --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --output "{OUT}/baseline_check.jsonl"

In [ ]:
!python -m phi35cfg.infer --mode rag --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --retrieval "{RAG}" --entropy "{ENTROPY}" --predictions-name {BASE_NAME} --top-percent 5 --output "{OUT}/rag5.jsonl"
!python -m phi35cfg.infer --mode expert_cfg --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --retrieval "{RAG}" --entropy "{ENTROPY}" --predictions-name {BASE_NAME} --top-percent 5 --output "{OUT}/cfg5.jsonl"

In [ ]:
# 전 문항 w/ RAG (가장 오래 걸린다)
!python -m phi35cfg.infer --mode rag --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --retrieval "{RAG}" --output "{OUT}/rag100.jsonl"

## 7. 결과표

In [ ]:
!python -m phi35cfg.evaluate report --dataset {DATASET} --data-root "{DATA_ROOT}" --baseline {BASELINE} --entropy "{ENTROPY}" --predictions-name {BASE_NAME} --rag100 "{OUT}/rag100.jsonl" --rag5 "{OUT}/rag5.jsonl" --cfg5 "{OUT}/cfg5.jsonl" --check-baseline "{OUT}/baseline_check.jsonl" --out-dir "{OUT}/results" --checkpoint-name {ADAPTER_NAME}

## 8. yes/no 불확실성 비교 (val, 확인용)
공식 정의는 D0(명세의 식 4)다. D1(대소문자 · 띄어쓰기 변형 합치기)과 D2(yes/no 이진 엔트로피)는 원인 확인용이며 식 4가 아니다. 게이팅과 명세는 바꾸지 않는다.

0 · 1절을 실행한 뒤 바로 이 절을 실행해도 된다(검색 파일 · GPU 점검 절은 필요 없다). `DATASET` 을 `vqarad` 와 `slake` 로 바꿔 두 번 실행하면(런타임 재시작) 마지막 셀이 보고서를 만든다.

In [ ]:
YESNO_ROOT = '/content/drive/MyDrive/phi35cfg_runs'       # 두 데이터셋의 OUT 이 이 아래에 있다

In [ ]:
# 현재 DATASET 의 val: 베이스라인(D0 와 오답 판정) + yes/no 첫 토큰 확률. 끊기면 다시 실행하면 이어서 한다
!python -m phi35cfg.infer --mode baseline --split val --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --output "{OUT}/val_release.jsonl"
!python -m phi35cfg.infer --mode yesno --split val --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --output "{OUT}/yesno_val.jsonl"

In [ ]:
# 두 데이터셋의 val 파일이 모두 있을 때 비교 (CPU, 몇 분)
need = [f'{YESNO_ROOT}/{d}/{f}' for d in ('vqarad', 'slake') for f in ('val_release.jsonl', 'yesno_val.jsonl')]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('아직 없는 파일:', missing)
else:
    !python -m phi35cfg.evaluate yesno --vqarad "{need[0]}" "{need[1]}" --slake "{need[2]}" "{need[3]}" --data-root "{DATA_ROOT}" --out "{YESNO_ROOT}/yesno_report.json"

## 9. 논문 Table 6 (파인튜닝 두 행: Label prob. · Entropy)
AUC 는 인용 문헌 [44] 의 **선택 정확도–커버리지 AUC** 이며 ROC-AUC 가 아니다. 무작위 순서의 값이 정확도와 같으므로 정확도와 함께 읽는다. 정의는 `docs/superpowers/specs/2026-11-28-table6-design.md` 에 고정돼 있다.

0 · 1절을 실행한 뒤 바로 이 절을 실행해도 된다. test 의 closed 문항만 돌며, 끊기면 다시 실행하면 이어서 한다. `DATASET` 마다 런타임을 다시 시작해 실행한다.

In [ ]:
!python -m phi35cfg.infer --mode table6 --dataset {DATASET} --image-root "{IMAGE_ROOT}" --data-root "{DATA_ROOT}" --base-dir {BASE_DIR} --adapter {ADAPTER} --output "{OUT}/table6_test.jsonl"

In [ ]:
!python -m phi35cfg.evaluate table6 --dataset {DATASET} --data-root "{DATA_ROOT}" --run "{OUT}/table6_test.jsonl" --out "{OUT}/table6_report.json"